# `ta_tools` exploration: AAPL since January 2023

A tour of the technical-analysis package on real data. Every indicator here comes from
`src.tools.ta_tools`, and so do the bars, through `ta_tools.load_bars`.

| Chart | Shows |
|---|---|
| 1 | Moving averages, Bollinger Bands (14 and 200), LinReg Candles, Trendlines with Breaks |
| 2 | RSI (14) |
| 3 | ATR (14) |

Chart 1 carries four overlays, too many to read at once, so buttons above it switch between
them. Prices are **as traded**, not dividend-adjusted (`auto_adjust=False`), as everywhere
in this repo.

In [1]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e ".[ta]"`.
# On Colab the install runs here, and the repo also goes on sys.path because an editable
# install is only picked up when the interpreter starts.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}. Update REPO above.')
    !pip install -q -e "{REPO}[ta]"
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

print(f'Running in Colab: {IN_COLAB}')

Running in Colab: False


## Data

In [2]:
import pandas as pd
import plotly.graph_objects as go

from src.tools import ta_tools

TICKER = 'AAPL'
START = '2023-01-01'

bars = ta_tools.load_bars(TICKER, START)
OPEN, HIGH, LOW, CLOSE = bars['open'], bars['high'], bars['low'], bars['close']

print(f'{TICKER}: {len(bars)} daily bars, {bars.index[0]:%Y-%m-%d} to {bars.index[-1]:%Y-%m-%d}')
bars.tail()

AAPL: 936 daily bars, 2023-01-03 to 2026-09-25


,open,high,low,close,volume
date,,,,,
2026-09-21,335.279999,339.640015,333.049988,338.980011,34999200.0
2026-09-22,340.140015,345.339996,338.750000,339.750000,40711800.0
2026-09-23,341.079987,341.799988,335.500000,337.019989,31658800.0
2026-09-24,336.720001,338.910004,334.299988,335.920013,24733100.0
2026-09-25,336.040009,341.670013,334.529999,341.070007,29950100.0


## Indicators

Each call below is one line. The two Pine ports run with the Pine scripts' default
inputs. `trendlines` runs in its default realtime mode, where every value uses only the
bars up to it; `backpaint=True` would reproduce the TradingView chart instead.

In [3]:
MA_LENGTH = 20

averages = {
    f'SMA {MA_LENGTH}': ta_tools.sma(CLOSE, MA_LENGTH),
    f'EMA {MA_LENGTH}': ta_tools.ema(CLOSE, MA_LENGTH),
    f'WMA {MA_LENGTH}': ta_tools.wma(CLOSE, MA_LENGTH),
    f'HMA {MA_LENGTH}': ta_tools.hma(CLOSE, MA_LENGTH),
    f'ALMA {MA_LENGTH}': ta_tools.alma(CLOSE, MA_LENGTH),
}
bands = {14: ta_tools.bb(CLOSE, 14), 200: ta_tools.bb(CLOSE, 200)}
lrc = ta_tools.linreg_candles(OPEN, HIGH, LOW, CLOSE)       # Pine defaults: 11, 11, SMA signal
tl = ta_tools.trendlines(HIGH, LOW, CLOSE)                  # Pine defaults: 14, 1.0, ATR slope
rsi = ta_tools.rsi(CLOSE, 14)
atr = ta_tools.atr(HIGH, LOW, CLOSE, 14)

print('BB 200 starts on ', bands[200]['bb_mid_200'].first_valid_index().date())
print('upper-line breaks:', int(tl['tl_upper_break'].sum()),
      '   lower-line breaks:', int(tl['tl_lower_break'].sum()))

BB 200 starts on  2023-10-18
upper-line breaks: 16    lower-line breaks: 13


In [4]:
# Chart styling shared by every figure. Price recedes to greys, leaving colour for the
# indicators, which take a validated categorical palette in fixed order.
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # blue, orange, aqua, yellow, magenta
CANDLES = dict(increasing_line_color='#b5b3aa', increasing_fillcolor='#e1e0d9',
               decreasing_line_color='#6f6d68', decreasing_fillcolor='#a9a79f')
INK = '#898781'

# Close the gaps weekends and market holidays would leave on a date axis.
HOLIDAYS = pd.bdate_range(bars.index[0], bars.index[-1]).difference(bars.index)
RANGEBREAKS = [dict(bounds=['sat', 'mon']), dict(values=HOLIDAYS.strftime('%Y-%m-%d').tolist())]


def style(fig, title=None, height=480):
    fig.update_layout(title=title, height=height, plot_bgcolor='white', paper_bgcolor='white',
                      hovermode='x unified', xaxis_rangeslider_visible=False,
                      legend=dict(traceorder='normal'), font=dict(color='#52514e'),
                      margin=dict(l=40, r=20, t=90, b=40))
    fig.update_xaxes(gridcolor='#e1e0d9', linecolor='#c3c2b7', rangebreaks=RANGEBREAKS)
    fig.update_yaxes(gridcolor='#e1e0d9', linecolor='#c3c2b7')
    return fig


def broken_at(line, restarts):
    """x and y for a line with a gap before each restart, so a new trendline does not join the old."""
    x, y = [], []
    for date, value, restart in zip(line.index, line, restarts):
        if restart:
            x.append(None)
            y.append(None)
        x.append(date)
        y.append(value)
    return x, y

## Chart 1: overlays on price

The buttons switch between four views:

- **Moving averages**: five kinds, all over 20 bars. HMA and ALMA hug the price; SMA lags most.
- **Bollinger Bands**: 14 bars and 200 bars, each ±2 standard deviations. BB 200 needs 199
  bars of history, so it starts in October 2023.
- **LinReg Candles**: each price replaced by its 11-bar linear-regression value, with the
  11-bar SMA signal line. Blue candles close above their open, orange below.
- **Trendlines**: a falling line from each pivot high, a rising line from each pivot low.
  ▲ marks a close above the falling line, ▼ a close below the rising line. Each line starts
  14 bars after its pivot, once the pivot is confirmed.

In [5]:
fig = go.Figure()
views = []          # the view each trace belongs to; the price candles belong to three


def add(trace, *view_names):
    fig.add_trace(trace)
    views.append(set(view_names))


PRICE_VIEWS = ('Moving averages', 'Bollinger Bands', 'Trendlines')
add(go.Candlestick(x=bars.index, open=OPEN, high=HIGH, low=LOW, close=CLOSE,
                   name=TICKER, **CANDLES), *PRICE_VIEWS)

for colour, (label, series) in zip(SERIES, averages.items()):
    add(go.Scatter(x=series.index, y=series, mode='lines', name=label,
                   line=dict(color=colour, width=1.5)), 'Moving averages')

for colour, fill, (length, frame) in zip(SERIES, ('rgba(42,120,214,0.08)', 'rgba(235,104,52,0.08)'),
                                         bands.items()):
    add(go.Scatter(x=frame.index, y=frame[f'bb_upper_{length}'], mode='lines',
                   name=f'BB {length} upper', line=dict(color=colour, width=1.5)), 'Bollinger Bands')
    add(go.Scatter(x=frame.index, y=frame[f'bb_lower_{length}'], mode='lines',
                   name=f'BB {length} lower', line=dict(color=colour, width=1.5),
                   fill='tonexty', fillcolor=fill), 'Bollinger Bands')
    add(go.Scatter(x=frame.index, y=frame[f'bb_mid_{length}'], mode='lines',
                   name=f'BB {length} basis', line=dict(color=colour, width=1.5, dash='dot')),
        'Bollinger Bands')

add(go.Candlestick(x=lrc.index, open=lrc['lrc_open'], high=lrc['lrc_high'], low=lrc['lrc_low'],
                   close=lrc['lrc_close'], name='LinReg candles',
                   increasing_line_color=SERIES[0], increasing_fillcolor=SERIES[0],
                   decreasing_line_color=SERIES[1], decreasing_fillcolor=SERIES[1]),
    'LinReg Candles')
add(go.Scatter(x=lrc.index, y=lrc['lrc_signal'], mode='lines', name='LinReg signal (SMA 11)',
               line=dict(color=SERIES[2], width=2)), 'LinReg Candles')

for colour, side, pivot in ((SERIES[0], 'upper', 'tl_pivot_high'),
                            (SERIES[1], 'lower', 'tl_pivot_low')):
    x, y = broken_at(tl[f'tl_{side}'], tl[pivot].notna())
    label = 'falling line (from pivot highs)' if side == 'upper' else 'rising line (from pivot lows)'
    add(go.Scatter(x=x, y=y, mode='lines', name=label, line=dict(color=colour, width=2)),
        'Trendlines')
up, down = tl['tl_upper_break'], tl['tl_lower_break']
add(go.Scatter(x=bars.index[up], y=LOW[up] * 0.98, mode='markers', name='break above ▲',
               marker=dict(symbol='triangle-up', size=11, color=SERIES[0],
                           line=dict(color='white', width=1))), 'Trendlines')
add(go.Scatter(x=bars.index[down], y=HIGH[down] * 1.02, mode='markers', name='break below ▼',
               marker=dict(symbol='triangle-down', size=11, color=SERIES[1],
                           line=dict(color='white', width=1))), 'Trendlines')

TITLES = {'Moving averages': f'{TICKER}: moving averages ({MA_LENGTH})',
          'Bollinger Bands': f'{TICKER}: Bollinger Bands (14 and 200, 2 SD)',
          'LinReg Candles': f'{TICKER}: Linear Regression Candles (11) and signal',
          'Trendlines': f'{TICKER}: Trendlines with Breaks (14, ATR slope, realtime)'}
buttons = [dict(label=name, method='update',
                args=[{'visible': [name in v for v in views]}, {'title.text': title}])
           for name, title in TITLES.items()]

first = next(iter(TITLES))
for trace, v in zip(fig.data, views):
    trace.visible = first in v
style(fig, TITLES[first], height=660)
# The title sits at the top and the view buttons in a row beneath it.
fig.update_layout(title=dict(y=0.98, yanchor='top'), margin=dict(t=110),
                  updatemenus=[dict(type='buttons', direction='right', buttons=buttons,
                                    x=0, xanchor='left', y=1.02, yanchor='bottom',
                                    bgcolor='white', bordercolor='#c3c2b7')])
fig.show()

## Chart 2: RSI (14)

Wilder's RSI over 14 bars. The dashed lines are the conventional 70 (overbought) and
30 (oversold) levels.

In [6]:
fig = go.Figure(go.Scatter(x=rsi.index, y=rsi, mode='lines', name='RSI 14',
                           line=dict(color=SERIES[0], width=1.5)))
for level in (70, 30):
    fig.add_hline(y=level, line=dict(color=INK, width=1, dash='dash'),
                  annotation_text=str(level), annotation_position='right')
style(fig, f'{TICKER}: RSI (14)', height=340)
fig.update_layout(margin=dict(t=50))
fig.update_yaxes(range=[0, 100])
fig.show()

## Chart 3: ATR (14)

Average true range over 14 bars in dollars, matching Pine's `ta.atr`. It is a price
distance, so it grows with the price as well as with volatility.

In [7]:
fig = go.Figure(go.Scatter(x=atr.index, y=atr, mode='lines', name='ATR 14',
                           line=dict(color=SERIES[0], width=1.5)))
style(fig, f'{TICKER}: ATR (14), $', height=340)
fig.update_layout(margin=dict(t=50))
fig.show()

## Latest values

The charts' last bar as a table, and the most recent trendline breaks.

In [8]:
last = bars.index[-1]
latest = {'Close': CLOSE[last]}
latest.update({label: series[last] for label, series in averages.items()})
for length, frame in bands.items():
    latest[f'BB {length} upper'] = frame[f'bb_upper_{length}'][last]
    latest[f'BB {length} lower'] = frame[f'bb_lower_{length}'][last]
latest.update({'LinReg close': lrc['lrc_close'][last], 'LinReg signal': lrc['lrc_signal'][last],
               'LinReg signal slope': lrc['lrc_slope'][last],
               'Falling trendline': tl['tl_upper'][last], 'Rising trendline': tl['tl_lower'][last],
               'RSI 14': rsi[last], 'ATR 14': atr[last]})
print(f'{TICKER} on {last:%Y-%m-%d}')
pd.Series(latest, name='value').round(2).to_frame()

AAPL on 2026-09-25


,value
Close,341.07
SMA 20,329.40
EMA 20,330.35
WMA 20,333.23
HMA 20,341.36
ALMA 20,336.85
BB 14 upper,347.76
BB 14 lower,316.97
BB 200 upper,342.09
BB 200 lower,233.44


In [9]:
breaks = pd.concat([
    pd.DataFrame({'break': 'above falling line', 'close': CLOSE, 'line': tl['tl_upper']})[up],
    pd.DataFrame({'break': 'below rising line', 'close': CLOSE, 'line': tl['tl_lower']})[down],
]).sort_index()
breaks.tail(10).round(2)

,break,close,line
date,,,
2025-06-13,below rising line,196.45,197.23
2025-07-28,above falling line,214.05,212.46
2025-08-01,below rising line,202.38,204.59
2025-11-21,above falling line,271.49,271.17
2025-12-17,below rising line,271.84,272.59
2026-02-04,above falling line,276.49,274.40
2026-03-12,below rising line,255.76,258.93
2026-04-15,above falling line,266.43,259.15
2026-07-02,above falling line,308.63,307.46
